# 6교시 · 통계분석 기초 with LLM
### — 어떤 검정을 쓸지는 물어보면 됩니다

**이 시간이 끝나면 할 수 있는 것**

1. 어떤 검정을 써야 할지 **AI에게 제대로 물어볼 수 있다**
2. 데이터를 올리지 않고도 **상황을 말로 설명**할 수 있다
3. 받은 코드를 돌리고, **결과를 해석**할 수 있다
4. **AI 가 틀렸을 때 알아챌 수 있다**

> 오늘 통계 공식은 하나도 외우지 않습니다.
> 대신 **무엇을 물어야 하는지**를 익힙니다.

In [ ]:
import pandas as pd
from scipy import stats

BASE = 'https://raw.githubusercontent.com/JasonWhiteLee/ak-data-analysis-basics/main/'

orders = pd.read_csv(BASE + 'superstore_orders.csv',
                     parse_dates=['Order Date', 'Ship Date']).drop_duplicates()

print(orders.shape)

## 그래프에 한글이 나오게 하기

뒤에서 산점도를 그립니다. 아래 두 셀을 먼저 실행하세요.

In [ ]:
!apt-get -qq install fonts-nanum > /dev/null

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm

fm.fontManager.addfont('/usr/share/fonts/truetype/nanum/NanumGothic.ttf')
plt.rc('font', family='NanumGothic')
plt.rc('axes', unicode_minus=False)

---
# 6-1. 검정 종류를 외울 필요가 없습니다

통계 검정은 종류가 많습니다. t-test, ANOVA, 카이제곱, 만-휘트니, 윌콕슨…
**어떤 상황에 무엇을 쓰는지 외우는 것이 예전에는 공부의 대부분이었습니다.**

지금은 그 부분을 물어보면 됩니다. **ChatGPT · Claude 에게 상황을 설명하면**
어떤 검정이 맞는지, 왜 그런지, 파이썬 코드까지 만들어 줍니다.
**휴대폰으로도 됩니다.**

그런데 회사 데이터는 대개 **올릴 수 없습니다.** 그래서 이렇게 합니다.

> **데이터를 올리는 대신, 데이터가 어떻게 생겼는지 말로 설명합니다.**

이게 오늘 배울 전부입니다. 그리고 말로 설명하려면
**한 줄이 무엇인지 · 열이 숫자인지 범주인지 · 집단이 몇 개인지**를
스스로 정리해야 합니다. 사실 그게 검정을 고르는 일의 본체입니다.

---
# 6-2. 그 전에 — 검정이 하는 일은 하나입니다

물어보기 전에 이것만 알고 있으면 됩니다.

## 두 집단을 재면 **언제나** 숫자가 다릅니다

A팀 평균과 B팀 평균을 재면 절대 똑같이 나오지 않습니다.
소수점 아래까지 보면 항상 다릅니다. **두 집단이 실제로는 아무 차이가 없어도 그렇습니다.**
우리가 재는 건 몇 명의 실제 사람들이고, 그 사람들이 누구냐에 따라 숫자가 흔들리기 때문입니다.

**그래서 "차이가 있느냐"는 질문은 의미가 없습니다.** 차이는 항상 있습니다.
진짜 질문은 이것입니다.

> **이 정도 차이가, 두 집단이 사실은 같은데도 그냥 흔들림으로 나올 만한 크기인가?**

## 동전으로 생각해 봅시다

멀쩡한 동전을 10번 던져 앞면이 6번 나왔습니다. 이상한 동전이라고 하시겠습니까?
아마 아닐 겁니다. 그런데 **100번에 60번**이면 슬슬 의심이 가고,
**1,000번에 600번**이면 거의 확실히 이상한 동전입니다.

**앞면 비율은 셋 다 60%로 같습니다.** 그런데 판단이 달라집니다.
던진 횟수가 많아질수록 "우연히 그렇게 나올 가능성"이 줄어들기 때문입니다.

**검정은 이 판단을 숫자로 해 주는 도구입니다.** 그 숫자가 p값입니다.

---
# 6-3. 좋은 질문의 다섯 가지 요소

데이터를 못 올리니 **말로 설명해야 합니다.** 다섯 가지를 빠뜨리지 마세요.

| | 무엇을 | 나쁜 예 | 좋은 예 |
|---|---|---|---|
| **①** | **상황** — 무슨 일을 하는 조직이고 왜 궁금한가 | (없음) | "유통사 영업기획입니다. 할인 정책을 바꿀지 검토 중입니다" |
| **②** | **알고 싶은 것** — 한 문장으로 | "분석해 주세요" | "할인한 주문이 그렇지 않은 주문보다 이익이 낮은지 알고 싶습니다" |
| **③** | **데이터 형태** — 한 줄이 무엇인지, 열 이름과 종류, 행 수 | "매출 데이터요" | "한 줄이 주문 품목 하나입니다. 1만 행. 열은 이익(숫자), 할인율(숫자), 지역(범주 4개)" |
| **④** | **비교 대상** — 집단이 몇 개인지, 짝지어져 있는지 | (없음) | "할인함 / 안 함 두 집단입니다. 서로 다른 주문이라 짝지어져 있지 않습니다" |
| **⑤** | **제약** — 못 하는 것, 쓰는 도구 | (없음) | "데이터는 못 올립니다. 파이썬 pandas 를 씁니다. 결측치가 조금 있습니다" |

## 그리고 **무엇을 달라고 할지**도 적습니다

- 어떤 검정이 맞는지 **와 그 이유**
- 파이썬 코드
- **결과를 어떻게 읽는지**
- **이 검정이 맞지 않게 되는 조건** ← 이걸 꼭 물어보세요

마지막 항목이 중요합니다. AI 는 웬만하면 자신 있게 답합니다.
**"언제 틀리냐"를 물어야 스스로 조건을 붙입니다.**

## 복사해서 쓰는 틀

```
[상황] 저는 ___ 에서 ___ 일을 합니다. 지금 ___ 를 결정하려고 합니다.

[알고 싶은 것] ___ 인지 알고 싶습니다.

[데이터 형태]
- 한 줄은 ___ 하나입니다. 총 ___ 행입니다.
- 열: ___(숫자), ___(범주, ___개), ___(날짜)
- 결측치: ___

[비교] ___ 와 ___ 를 비교하려 합니다. 두 집단은 서로 (짝지어져 있음 / 다른 대상임).

[제약] 데이터는 올릴 수 없습니다. ___ 로 작업합니다.

[요청]
1. 어떤 검정이 맞습니까? 왜 그렇습니까?
2. 파이썬 코드를 주세요.
3. 결과를 어떻게 읽습니까?
4. 이 검정이 맞지 않게 되는 경우는 언제입니까?
```

---
# 6-4. 실습 — 실제로 물어보기

**휴대폰으로 해도 됩니다.** ChatGPT 나 Claude 앱을 열고 아래 질문을 보내 보세요.
오늘 쓰는 데이터로 만든 예시입니다.

```
[상황] 문구·가구 유통사의 영업기획 담당입니다.
할인 정책을 손볼지 검토하고 있습니다.

[알고 싶은 것] 할인을 한 주문이 할인을 안 한 주문보다
건당 이익이 낮은지 알고 싶습니다.

[데이터 형태]
- 한 줄은 주문에 담긴 품목 하나입니다. 총 10,194 행입니다.
- 열: 이익(숫자), 할인율(0~0.8 숫자), 지역(범주 4개), 대분류(범주 3개)
- 결측치: 이익·할인율에는 없습니다.

[비교] 할인율이 0보다 큰 주문과 0인 주문을 비교하려 합니다.
서로 다른 주문이라 짝지어져 있지 않습니다.

[제약] 데이터는 올릴 수 없습니다. 파이썬 pandas 로 작업합니다.

[요청]
1. 어떤 검정이 맞습니까? 왜 그렇습니까?
2. 파이썬 코드를 주세요.
3. 결과를 어떻게 읽습니까?
4. 이 검정이 맞지 않게 되는 경우는 언제입니까?
```

> **직접 해 보세요.** 그리고 옆 사람과 답을 비교해 보세요.
> 같은 질문인데 답이 조금씩 다를 수 있습니다. 그것도 정보입니다.

## 대개 이렇게 답합니다

> **독립표본 t-검정(independent two-sample t-test)** 을 쓰면 됩니다.
> 두 집단이 서로 다른 대상이고, 비교하려는 값(이익)이 숫자이기 때문입니다.
> 두 집단의 분산이 다를 수 있으므로 **Welch 방식**(`equal_var=False`)을 권합니다.

그리고 아래 같은 코드를 줍니다.

---
# 6-5. 받은 코드를 돌려 봅니다

AI 가 준 코드는 대개 이런 모양입니다. **직접 실행해 보세요.**

In [ ]:
discounted     = orders[orders['Discount'] > 0]['Profit']
not_discounted = orders[orders['Discount'] == 0]['Profit']

print('할인함   평균 이익: {:>8.2f}   ({:,}건)'.format(discounted.mean(), len(discounted)))
print('할인안함 평균 이익: {:>8.2f}   ({:,}건)'.format(not_discounted.mean(), len(not_discounted)))

In [ ]:
result = stats.ttest_ind(discounted, not_discounted, equal_var=False)

print('p값:', result.pvalue)

## 결과

| | 평균 이익 |
|---|---|
| 할인함 | **-6.53** |
| 할인 안 함 | **+66.34** |

p값은 `3.7e-56` 입니다. **0 을 55개 붙인 것보다 작은 수**라는 뜻입니다.

> **"두 집단이 사실 같다고 쳤을 때, 이만큼 차이가 우연히 날 확률"** 이 그만큼 작습니다.
> 우연으로 보기는 어렵다는 뜻입니다.

---
# 6-6. 해석도 물어보면 됩니다 — 다만 이 셋은 직접 확인하세요

결과를 그대로 붙여 넣고 **"이걸 어떻게 읽습니까"** 라고 물어도 됩니다.
잘 설명해 줍니다.

**그런데 AI 는 틀린 답도 자신 있게 합니다.** 그래서 아래 세 가지는
직접 알고 있어야 합니다. 이게 오늘 통계에서 외울 전부입니다.

## ① p값은 "내 주장이 맞을 확률"이 아닙니다

| 흔한 오해 | 실제 |
|---|---|
| "p=0.03 이면 내 주장이 맞을 확률 97%" | **아닙니다.** p값은 내 주장의 확률이 아닙니다 |
| "p=0.4 면 두 집단이 같다는 뜻" | **아닙니다.** "다르다고 말할 근거가 부족하다" 일 뿐입니다 |
| "p 가 작으면 차이가 크다" | **아닙니다.** 차이 크기와 p값은 다른 이야기입니다 |

p값의 뜻은 하나뿐입니다 —
**두 집단이 사실 같다고 쳤을 때, 지금 본 정도의 차이가 우연히 나올 확률.**

## ② 유의하다 ≠ 중요하다

데이터가 아주 많으면 **아무리 작은 차이도 p값이 작게 나옵니다.**
직접 확인해 보세요.

In [ ]:
import numpy as np
np.random.seed(0)

for n in [100, 1000, 100000]:
    A = np.random.normal(100,   10, n)
    B = np.random.normal(100.5, 10, n)      # 실제 차이는 늘 0.5
    p = stats.ttest_ind(A, B, equal_var=False).pvalue
    print('표본 {:>6}개씩 → 실제 차이 0.5 → p값 {:.4f}'.format(n, p))

**셋 다 실제 차이는 0.5 로 같습니다.** 그런데 표본이 커질수록 p값만 작아집니다.

> 그래서 보고할 때는 **p값과 실제 차이 크기를 함께** 적습니다.
> "할인한 주문은 건당 이익이 **73 낮습니다** (p < 0.001)" — 이렇게요.

## ③ 상관은 인과가 아닙니다

이건 6-8 에서 따로 봅니다. **오늘 가장 중요한 이야기입니다.**

---
# 6-7. 상관 — 두 숫자가 같이 움직이는가

검정과 함께 자주 쓰는 것이 **상관계수**입니다. 이것도 한 줄이면 나옵니다.

In [ ]:
print(round(orders['Discount'].corr(orders['Profit']), 3))

## 상관계수 읽는 법

**-1 에서 +1 사이**의 값입니다.

| 값 | 뜻 |
|---|---|
| **+1 에 가깝다** | 한쪽이 오르면 다른 쪽도 오른다 |
| **0 에 가깝다** | 직선 관계가 약하다 — *관계가 없다는 뜻은 아닙니다* |
| **-1 에 가깝다** | 한쪽이 오르면 다른 쪽은 내린다 |

할인율과 이익은 **-0.219** 입니다. 음의 관계이긴 한데 **약합니다.**
"할인하면 이익이 준다"고 단정하기에는 부족한 숫자입니다.

**숫자만 보지 말고 그림을 함께 보세요.**

In [ ]:
display = orders[orders['Profit'].between(-500, 500)]

display.plot(kind='scatter', x='Discount', y='Profit', alpha=0.15, figsize=(8, 4))
plt.axhline(0, color='red', linewidth=1)
plt.title('할인율과 이익 — 상관 -0.219 의 실제 모습')
plt.show()

**깔끔한 선이 아닙니다.** 점이 사방에 흩어져 있습니다.
할인이 0 인 쪽은 대부분 빨간 선 위(이익 +)에 있고,
할인이 커질수록 아래로 내려가는 점이 많아지는 정도입니다.

> 상관계수 하나만 보고 "관계가 있다/없다"를 말하지 마세요. **그림을 보세요.**

---
# 6-8. 상관은 인과가 아닙니다

**오늘 가장 중요한 절입니다.**

검정까지 마치면 이런 보고서를 쓸 수 있습니다.
"A 와 B 는 관계가 있고, 검정 결과 우연이 아닙니다."

**그다음에 거의 항상 이런 말이 따라옵니다** — "그럼 A 를 바꾸면 B 가 바뀌겠네요?"

**아닙니다.** 왜 아닌지 실제 데이터로 보겠습니다.

### 새 데이터 — 브라질 온라인 커머스 (Olist)

| | |
|---|---|
| **무엇** | 브라질 전자상거래 플랫폼 Olist 의 공개 데이터 |
| **기간** | 2016-09 ~ 2018-10 |
| **크기** | 주문 99,441건 · 리뷰 99,224건 |

**이번에 쓰는 두 표**

| 표 | 주요 열 |
|---|---|
| `olist_orders_dataset.csv` | `order_id` · `order_delivered_customer_date`(실제 도착일) ·<br>`order_estimated_delivery_date`(약속한 도착일) |
| `olist_order_reviews_dataset.csv` | `order_id` · `review_score`(별점 1~5) |

> 약속한 날짜와 실제 도착일이 함께 있어서 **배송이 늦었는지** 알 수 있고,
> 거기에 별점을 붙여 볼 수 있습니다. 파일이 커서 몇 초 걸립니다.

In [ ]:
oo = pd.read_csv(BASE + 'olist/olist_orders_dataset.csv',
                 parse_dates=['order_delivered_customer_date',
                              'order_estimated_delivery_date'])
rv = pd.read_csv(BASE + 'olist/olist_order_reviews_dataset.csv')

print('주문:', oo.shape, ' 리뷰:', rv.shape)

In [ ]:
delivery = oo.merge(rv[['order_id', 'review_score']], on='order_id')

# 실제 도착일 - 약속한 도착일 (양수면 늦은 것)
delivery['delay_days'] = (delivery['order_delivered_customer_date']
                          - delivery['order_estimated_delivery_date']).dt.days

delivery = delivery.dropna(subset=['delay_days', 'review_score'])
delivery['is_late'] = delivery['delay_days'] > 0

delivery.groupby('is_late')['review_score'].agg(['mean', 'count']).round(2)

## 결과

| | 평균 별점 | 건수 |
|---|---|---|
| **약속한 날짜를 지킴** | **4.29** | 89,949 |
| **약속한 날짜를 넘김** | **2.27** | 6,410 |

**별점이 2점 넘게 차이납니다.** 5점 만점에서 2점은 아주 큰 차이입니다.
검정해 보면 p값도 표시할 수 없을 만큼 작게 나옵니다.

---
# 그럼 이렇게 보고해도 됩니까?

> ### **"배송을 빨리 하면 별점이 오릅니다. 물류에 투자합시다."**

## 답: **알 수 없습니다.**

데이터가 말해 준 것은 **"늦은 주문의 별점이 낮다"** 입니다.
**"늦게 배송했기 때문에 별점이 낮아졌다"** 는 말이 아닙니다.

## 늦게 배송된 주문이 어떤 주문인지 생각해 봅시다

배송이 늦은 주문은 아무 주문이나가 아닙니다. **애초에 다른 주문들입니다.**

| 늦은 이유 | 별점이 낮은 다른 이유 |
|---|---|
| **먼 지역**이라 오래 걸렸다 | 먼 지역은 운송 중 파손도 많다 |
| **재고가 없어서** 늦게 발송됐다 | 재고 관리가 부실한 판매자의 상품이다 |
| **무겁거나 큰 물건**이라 늦었다 | 조립·설치가 번거로운 상품이다 |
| **영세 판매자**라 처리가 늦었다 | 포장·응대도 부실했다 |

**이 중 무엇이든 배송 속도와 상관없이 별점을 낮출 수 있습니다.**

## 인과를 말하려면

> ### **"그 주문들을 만약 제때 배송했다면, 별점이 올랐을까?"**

이 질문에 답할 수 있어야 합니다. **그런데 그 주문들은 이미 늦게 배송됐습니다.**
"만약 그러지 않았다면"의 세계는 관찰할 수 없습니다.

그래서 **A/B 테스트**를 합니다. 조건이 비슷한 주문을 둘로 나눠
한쪽만 빠르게 보내 보는 것입니다. 그러면 다른 조건이 같아지므로
차이를 배송 속도 탓으로 돌릴 수 있습니다.

> **이것만은 AI 에게 맡기지 마세요.**
> AI 는 "상관이 있습니다" 까지는 잘 알려 주지만,
> **당신이 인과라고 쓰면 그대로 통과시킵니다.**

---
# 6-9. 어떤 검정을 쓰는지 — 참고용 표 한 장

물어보면 되지만, **대략 감은 있는 게 좋습니다.** 외우지 마세요.

| 알고 싶은 것 | 흔히 쓰는 방법 |
|---|---|
| 두 집단의 **평균** 차이 | t-검정 (t-test) |
| 세 집단 이상의 평균 차이 | 분산분석 (ANOVA) |
| **범주끼리** 관련이 있는지 (지역 × 반품여부) | 카이제곱 검정 |
| 두 **숫자**가 같이 움직이는지 | 상관계수 |
| 여러 요인이 **각각** 얼마나 영향을 주는지 | 회귀분석 |

> 이 표는 "이런 게 있구나" 정도로 보세요.
> 실제로는 **상황을 설명하고 추천받는 편이 훨씬 정확합니다** —
> 짝지어진 데이터인지, 분포가 치우쳤는지 같은 조건까지 봐 주기 때문입니다.

---
# 정리 — 오늘 한 것

## 코드

| 하는 일 | 코드 |
|---|---|
| 두 집단 만들기 | `a = df[조건]['값']` · `b = df[~조건]['값']` |
| 평균 비교 검정 | `stats.ttest_ind(a, b, equal_var=False)` |
| p값 꺼내기 | `결과.pvalue` |
| 두 숫자의 상관 | `df['열1'].corr(df['열2'])` |
| 관계 그려 보기 | `df.plot(kind='scatter', x=, y=)` |

## 남길 것 세 가지

1. **어떤 검정을 쓸지는 물어보면 됩니다** — 대신 상황·데이터 형태·비교 대상을 **말로** 설명할 수 있어야 합니다
2. **p값은 내 주장이 맞을 확률이 아닙니다** — 그리고 유의하다고 중요한 것도 아닙니다
3. **상관은 인과가 아닙니다** — 이것만은 AI 가 대신 판단해 주지 않습니다

---

### 다음 시간

오늘 하루 배운 것을 **처음부터 끝까지 한 번에** 해 봅니다.
그리고 이렇게 묻습니다 — **그래서 무엇을 다르게 할 것인가?**

---
# 연습문제

**빈칸(`____`)을 채우고 실행**하세요.

### 문제 1. 할인한 주문과 안 한 주문의 이익 평균을 각각 구하세요.

In [ ]:
a = orders[orders['Discount'] > 0]['Profit']
b = orders[orders['Discount'] == 0]['Profit']

print('할인함  :', round(a.mean(), 2))
print('할인안함:', round(b.mean(), 2))

### 문제 2. 두 집단의 평균 차이를 검정하세요.

In [ ]:
print('p값:', stats.ttest_ind(a, b, equal_var=False).pvalue)

### 문제 3. 고객 유형(Consumer vs Corporate)으로 같은 검정을 하세요.

In [ ]:
a2 = orders[orders['Segment'] == 'Consumer']['Profit']
b2 = orders[orders['Segment'] == 'Corporate']['Profit']

print('평균:', round(a2.mean(), 2), 'vs', round(b2.mean(), 2))
print('p값 :', stats.ttest_ind(a2, b2, equal_var=False).pvalue)

### 문제 4. 매출과 이익의 상관계수를 구하세요.

In [ ]:
print(round(orders['Sales'].corr(orders['Profit']), 3))

### 문제 5. 수량과 이익의 상관계수를 구하세요.

In [ ]:
print(round(orders['Quantity'].corr(orders['Profit']), 3))

### 문제 6. 세 숫자 열의 상관계수 표를 한 번에 만드세요.

In [ ]:
orders[['Sales', 'Quantity', 'Discount']].corr().round(3)

---
### 마지막 문제 — 코드가 아닙니다

**여러분 업무에서 실제로 궁금한 것 하나**를 골라,
6-3 의 틀에 맞춰 질문을 써 보세요. 그리고 **휴대폰으로 물어보세요.**

데이터를 올리지 않고도 답을 받을 수 있는지,
받은 답이 말이 되는지 확인해 보는 것이 오늘의 마지막 실습입니다.